# Run an open-source model on the hallucination question set

This notebook runs **Qwen2.5-1.5B-Instruct** from the Hugging Face Hub (PyTorch + Transformers) on all 36 questions, scores the answers with the same GPT-4o-mini judge used for the API models, and computes the same metrics.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*, then Save. (It also works on CPU, just slower.)
2. Add your OpenAI key for the judge: click the **key icon** in the left sidebar → *Add new secret* → name it `OPENAI_API_KEY`, paste the key, and switch on *Notebook access*.

Then run the cells in order (*Runtime → Run all*). The whole thing takes about 10–15 minutes on a T4 GPU and costs a few cents in judge calls.

In [ ]:
!git clone -q https://github.com/Afia-Zaman/llm-hallucination-healthcare-legal.git
%cd llm-hallucination-healthcare-legal
!pip install -q -r requirements-open-source.txt

## 1. Generate answers locally with the open-source model

In [ ]:
!python src/run_open_source_model.py --model Qwen/Qwen2.5-1.5B-Instruct

If the summary lists rows that were cut off, run the cell above again; it only redoes those rows. If they keep getting cut off, add `--max-new-tokens 2048`.

## 2. Score the answers with the LLM judge

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

!python src/score_responses.py --raw results/open_source/raw_responses.csv --out results/open_source/scored_responses.csv

## 3. Compute the metrics

In [ ]:
!python src/analyze_results.py --scored results/open_source/scored_responses.csv --out results/open_source/summary_metrics.csv

## 4. Download the results

This downloads `open_source_results.zip` with the three CSVs. Upload them to your GitHub repo in a `results/open_source/` folder.

Before publishing, read every non-perfect score in `scored_responses.csv` and compare it with the raw answer, the same way the API results were checked. Put any correction in `human_corrected_score` with a reason in `correction_note`, then re-run step 3.

In [ ]:
!cd results && zip -q -r ../open_source_results.zip open_source
from google.colab import files
files.download("open_source_results.zip")